In [1]:
import torch
from model.model import EncoderDecoderDAG
from utils.data import TranslateDataset, collate_fn, process_data
from torch.utils.data import DataLoader
from utils.load_tokenizer import load_tokenizer
from torch.optim import Adam
from typing import Tuple
from utils.fix_probs import fix_probs
from losses.dag_loss import masking
from tqdm.notebook import tqdm
from matplotlib import pyplot as plt
from utils.checkpoint import try_loading, epoch_resume, save_checkpoint
from utils.decoding import greedy_decoding, lookahead

g:\Projects\Visual Studio Code\LMTests\lmtest\lib\site-packages\transformers\utils\hub.py:123: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
tokenizer, vocab_size = load_tokenizer()

In [3]:
pad_idx = tokenizer.pad_token_id
eos_idx = tokenizer.eos_token_id

In [4]:
factor = 4
emb_size = 256
num_heads = 8
max_seq_len = 100
max_vertices = max_seq_len * factor

In [5]:
layers = [(2,2), 1, (1,1)]
out_layers = 1

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
device

device(type='cuda')

In [8]:
def create_model_fallback_fn() -> Tuple[EncoderDecoderDAG, Adam]:
    model = EncoderDecoderDAG(vocab_size, emb_size, num_heads, max_seq_len, max_vertices, layers, out_layers)
    model.to(device)
    optm = Adam(model.parameters(), lr=1e-3)
    return model, optm
    

In [9]:
checkpoint_dir = "./checkpoints"
checkpoint_name = "naivedag.pt"

In [10]:
model_class = EncoderDecoderDAG
optm_class = Adam

In [11]:
model, optm, losses, log_dir, tokens_seen = try_loading(checkpoint_dir, checkpoint_name, model_class, optm_class, device, create_model_fallback_fn)

Resuming, have seen 110,000 epochs and 40,573,858 tokens
Have 26704968 trainable parameters
Logging to runs/run_at_2023-12-24_13-34-45


In [12]:
en_test = "How are you"

In [13]:
encoded = tokenizer(en_test, return_tensors="pt").input_ids.to(device)

In [14]:
encoded = torch.nested.nested_tensor([encoded])

C:\Users\John\AppData\Local\Temp\ipykernel_1676\363011816.py:1: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ..\aten\src\ATen\NestedTensorImpl.cpp:180.)
  encoded = torch.nested.nested_tensor([encoded])


In [17]:
encoded = torch.nested.to_padded_tensor(encoded, pad_idx, (1, 1, 100))

In [19]:
encoded = encoded.squeeze(0)

In [20]:
encoded

tensor([[  904,    44,    37,     0, 65000, 65000, 65000, 65000, 65000, 65000,
         65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000,
         65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000,
         65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000,
         65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000,
         65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000,
         65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000,
         65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000,
         65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000,
         65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000, 65000]],
       device='cuda:0')

In [21]:
batch_size, l = encoded.shape
decoder_tokens = torch.arange(0, l * factor).unsqueeze(0).expand(batch_size, -1).to(device)
target_lens, vertex_lens, token_mask, vertex_mask = process_data(encoded, pad_idx, factor)
log_transition_probs, log_emission_probs = model(encoded, decoder_tokens, token_mask, vertex_mask)
mask = masking(log_transition_probs, vertex_lens)

In [22]:
log_transition_probs[0][2]

tensor([-3.2759e+01, -3.0401e+01, -3.5838e+01, -3.1232e+01, -3.4264e+01,
        -3.5109e+01, -3.8150e+01, -3.7055e+01, -3.2499e+01, -3.1628e+01,
        -3.6712e+01, -2.1815e+01, -2.0848e+01, -3.2377e+01, -2.8244e+01,
        -1.4305e-06, -3.3851e+01, -3.4409e+01, -2.8146e+01, -1.3516e+01,
        -4.1573e+01, -3.2958e+01, -3.0218e+01, -1.8441e+01, -3.1088e+01,
        -3.2418e+01, -3.3154e+01, -1.8089e+01, -3.2108e+01, -2.7421e+01,
        -3.3553e+01, -3.3421e+01, -4.1103e+01, -3.1016e+01, -3.0226e+01,
        -1.7152e+01, -3.3768e+01, -3.8873e+01, -2.9536e+01, -3.2084e+01,
        -3.3697e+01, -3.1282e+01, -3.2237e+01, -2.3795e+01, -3.1709e+01,
        -3.9713e+01, -2.7756e+01, -3.3414e+01, -3.2562e+01, -3.3920e+01,
        -3.5024e+01, -3.1780e+01, -4.2405e+01, -2.5855e+01, -3.3598e+01,
        -3.4127e+01, -3.0900e+01, -3.7717e+01, -2.9131e+01, -2.7764e+01,
        -3.4691e+01, -3.1804e+01, -3.2068e+01, -2.9983e+01, -3.5609e+01,
        -3.0702e+01, -3.0773e+01, -2.6327e+01, -3.1

In [23]:
#flog_transition_probs = log_transition_probs.masked_fill(mask !=0, float('-inf'))
flog_transition_probs = fix_probs(log_transition_probs, mask)
b1_transitions = flog_transition_probs[0]
b1_emissions = log_emission_probs[0]

In [24]:
b1_transitions[2]

tensor([       -inf,        -inf,        -inf, -1.6022e+01, -1.6022e+01,
        -1.6022e+01, -1.6022e+01, -1.6022e+01, -1.6022e+01, -1.6022e+01,
        -1.6022e+01, -1.6019e+01, -1.6014e+01, -1.6022e+01, -1.6022e+01,
        -1.3113e-06,        -inf,        -inf,        -inf,        -inf,
               -inf,        -inf,        -inf,        -inf,        -inf,
               -inf,        -inf,        -inf,        -inf,        -inf,
               -inf,        -inf,        -inf,        -inf,        -inf,
               -inf,        -inf,        -inf,        -inf,        -inf,
               -inf,        -inf,        -inf,        -inf,        -inf,
               -inf,        -inf,        -inf,        -inf,        -inf,
               -inf,        -inf,        -inf,        -inf,        -inf,
               -inf,        -inf,        -inf,        -inf,        -inf,
               -inf,        -inf,        -inf,        -inf,        -inf,
               -inf,        -inf,        -inf,     

In [25]:
torch.argmax(b1_transitions, dim=1)

tensor([ 7, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0, 

In [19]:
torch.argmax(b1_emissions, dim=1)

tensor([65001,  1068,  9737,   146,   146, 26030,   146,  9737,  1068, 26030,
          146,     0,     0,  9737,     0,     0], device='cuda:0')

In [26]:
decoded = greedy_decoding(b1_transitions, b1_emissions, eos_idx)

In [27]:
tokenizer.decode(decoded)

'<s> 年轻</s>'

In [28]:
decoded2 = lookahead(b1_transitions, b1_emissions, eos_idx)

In [29]:
tokenizer.decode(decoded2)

'<s> </s>'